# Очень продвинутый Python

### Менеджеры контекста (`with`)

Часто в программировании нам нужно выделить какой-то ресурс, поработать с ним, а затем обязательно освободить (закрыть файл, отключиться от базы данных). Если в процессе работы произойдет ошибка, мы можем забыть освободить ресурс. Конструкция `with` (менеджер контекста) гарантирует, что ресурс будет закрыт в любом случае.

Я написал только необходимую на мой взгляд информацию, при желании копнуть глубже, можете посмотреть видео про менеджеры контекста: https://www.youtube.com/watch?v=FkhnVkl0EgMэ

Самый частый пример — чтение и запись файлов. Чтобы файлом можно было пользоваться, можно открыть его с помощью `open`, и потом обязательно закрыть через `close`, но контекстный менеджер контролирует это за нас.

In [ ]:
with open('example.txt', 'w', encoding='utf-8') as f:
    f.write('Привет, мир!')

# Как только мы вышли из блока with, файл f автоматически закрылся.
# Нам не нужно писать f.close()

JSON — это популярный текстовый формат обмена данными (например, при общении фронтенда и бекенда, или для сохранения конфигов). В Python есть встроенная библиотека `json`, которая отлично работает в связке с менеджерами контекста.

In [1]:
import json

data = {'name': 'Иван', 'age': 30, 'skills': ['Python', 'SQL']}

# Сохраняем в файл (dump)
with open('data.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

# Читаем из файла (load)
with open('data.json', 'r', encoding='utf-8') as f:
    loaded_data = json.load(f)
    
print("Навыки:", loaded_data['skills'])

Навыки: ['Python', 'SQL']


Мы можем создавать собственные менеджеры контекста. Для этого класс должен реализовывать два магических метода:
- `__enter__` — вызывается при входе в блок `with` (его результат передается в переменную после `as`).
- `__exit__` — вызывается при выходе (здесь мы закрываем соединения, даже если внутри блока произошла ошибка).

In [2]:
class MyDatabaseConnection:
    def __enter__(self):
        print('Подключаемся к БД...')
        return 'Объект подключения (например, курсор)'
        
    def __exit__(self, exc_type, exc_value, traceback):
        # не вдавайтесь в эти аргументы, они нужны для обработки ошибок
        print('Закрываем соединение с БД...')

with MyDatabaseConnection() as db:
    print(f'Работаем: {db}')

Подключаемся к БД...
Работаем: Объект подключения (например, курсор)
Закрываем соединение с БД...


#### Задание

Напишите менеджер контекста `Timer`, который измеряет время выполнения блока кода. При входе в блок он должен запоминать текущее время, а при выходе — печатать, сколько секунд выполнялся код.

*Подсказка: используйте модуль `time` и функцию `time.time()` для получения текущего времени.*

In [9]:
import time

class Timer:
    def __init__(self):
        self._start_time = 0
        self._finish_time = 0
    
    def __enter__(self):

        self._start_time = time.time()

    
    def __exit__(self, exc_type, exc_value, traceback):
        self._finish_time = time.time()
        return print(f"Код выполнялся примерно {round(self._finish_time - self._start_time, 1)} секунд")


with Timer():
    print("Код выполняется...")
    time.sleep(1)
# Ожидаемый вывод: Код выполнялся примерно 1.0 секунд

Код выполняется...
Код выполнялся примерно 1.0 секунд


### Работа с функциями как с объектами

В Python функции — это объекты. Это значит, что вы можете делать с функциями то же самое, что и с обычными переменными: сохранять их, передавать как аргументы в другие функции и возвращать как результат.

In [ ]:
def say_hello(name):
    return f'Привет, {name}!'

# Присваиваем функцию переменной (без скобок!)
my_func = say_hello
print(my_func('Алексей'))

#### Анонимные функции (`lambda`)

Иногда нужна маленькая функция на один раз, и писать полноценный `def` слишком громоздко. В таких случаях используют `lambda` (анонимные функции).
Она пишется в одну строку: 

`lambda аргументы: выражение_которое_вернется`

In [10]:
square = lambda x,y: x * y
print(square(5,5))

25


#### `sorted`

Передача функции как аргумента невероятно полезна при сортировке. Встроенная функция `sorted()` может принимать аргумент `key`. Ему передается функция, которая будет применена к каждому элементу перед тем, как сравнивать их.

In [11]:
words = ['яблоко', 'кот', 'автомобиль', 'дом']

# Сортируем по длине слова (передаем функцию len)
sorted_words = sorted(words, key=len)
print("По длине:", sorted_words)

# Сортируем по последней букве (используя lambda)
sorted_by_last = sorted(words, key=lambda word: word[-1])
print("По последней букве:", sorted_by_last)

По длине: ['кот', 'дом', 'яблоко', 'автомобиль']
По последней букве: ['дом', 'яблоко', 'кот', 'автомобиль']


#### `filter` и `map`

- `map(func, iterable)` применяет функцию `func` к каждому элементу списка.
- `filter(func, iterable)` оставляет только те элементы, для которых `func` вернула `True`.

Обе эти функции возвращают специальные итераторы, поэтому чтобы увидеть результат, их обычно оборачивают в `list()`.

In [ ]:
numbers = [1, 2, 3, 4, 5]

# Умножаем все числа на 2
doubled = list(map(lambda x: x * 2, numbers))
print('Map:', doubled)

# Оставляем только четные числа
evens = list(filter(lambda x: x % 2 == 0, numbers))
print('Filter:', evens)

#### Задание

Напишите lambda-функцию, которая принимает одно число и возвращает его квадрат.
Присвойте эту функцию переменной `square`.

In [12]:
square = lambda x: x**2 # TODO ваш код здесь

assert square(5) == 25

Отфильтруйте список чисел так,
чтобы в нем остались только четные числа.

In [14]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
even_numbers = list(filter(lambda x: x %2 == 0 , numbers))  # TODO Ваш код вместо `lambda x: None`

assert even_numbers == [2, 4, 6, 8, 10]

Ниже дан список словарей с информацией о людях.
1. Отфильтруйте список, оставив только тех, кто старше 18 лет.
2. Отсортируйте полученный результат по имени в алфавитном порядке.

In [23]:
people = [
    {'name': 'Иван', 'age': 15},
    {'name': 'Анна', 'age': 22},
    {'name': 'Борис', 'age': 30},
    {'name': 'Мария', 'age': 17}
]

result = sorted(list(filter(lambda x: x['age'] > 18, people)), key=lambda x: x['name'])


assert result == [{'name': 'Анна', 'age': 22}, {'name': 'Борис', 'age': 30}]

### Декораторы

Декоратор — это функция, которая принимает другую функцию, расширяет или меняет её поведение, и возвращает новую (обернутую) функцию. Декораторы позволяют добавлять новый функционал к уже написанным функциям, не меняя их исходный код. А ещё про них часто спрашивают на собесах.

Для удобства используется синтаксис `@имя_декоратора`.

In [ ]:
def my_decorator(func):
    # wrapper - это обертка вокруг оригинальной функции
    def wrapper():
        print('--> Что-то делаем ДО вызова функции')
        func()
        print('<-- Что-то делаем ПОСЛЕ вызова функции')
    return wrapper

@my_decorator
def say_hi():
    print('Всем привет!')

say_hi()

Чтобы декоратор мог работать с функциями, принимающими аргументы, внутри `wrapper` обычно используют `*args` и `**kwargs`.

Подробнее про декоратор в видео https://www.youtube.com/watch?v=QxTi0X8iZ7s

#### Задание

Напишите декоратор `@debug`, который перед вызовом функции печатает, с какими аргументами она была вызвана, выполняет функцию, и перед возвратом печатает, что она вернула.

In [26]:
import functools
def debug(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"функция вернула резульат {result}")
        return result
    return wrapper


@debug
def add(a, b):
    return a + b

add(5, 3)

# Ожидаемый вывод:
# Вызываем add с аргументами: (5, 3) {}
# Функция вернула: 8

функция вернула резульат 8


8

### Итераторы

В Python часто нужно перебирать последовательности элементов.
- **Итерируемый объект (Iterable)** — то, по чему можно пройтись в цикле `for` (список, строка, кортеж, словарь). Главная проблема обычных коллекций (например, списков) в том, что они **загружают все свои элементы в память сразу**. Если список состоит из миллионов строк, это может занять всю оперативную память.
- **Итератор (Iterator)** — специальный объект, который решает эту проблему. Он не хранит все элементы в памяти, а «выдает» их строго по одному (лениво) с помощью встроенной функции `next()`. Он просто помнит, на каком шаге остановился.

In [6]:
my_list = [10, 20, 30]
my_iterator = iter(my_list)

print(next(my_iterator))
print(next(my_iterator))
print(next(my_iterator))
# Если вызвать next() еще раз, будет ошибка StopIteration

10
20
30


#### Свой итератор (магические методы `__iter__` и `__next__`)

Просто чтобы на собесе отвечая про итератор сказать что можно сделать итератор определив в классе эти методы, как работает под капотом можете скипать.

Мы можем сделать любой свой класс итератором. Для этого в нем нужно реализовать два магических метода:
1. `__iter__(self)` — должен возвращать сам объект-итератор (обычно просто пишут `return self`).
2. `__next__(self)` — должен вычислять и возвращать следующий элемент. Если элементы закончились, он должен вызывать ошибку `StopIteration` (это сигнал для цикла `for`, что нужно остановиться).

In [7]:
class MyRange:
    def __init__(self, start, end):
        self.current = start
        self.end = end
        
    def __iter__(self):
        return self
        
    def __next__(self):
        if self.current >= self.end:
            raise StopIteration  # Сигнал об окончании
            
        value = self.current
        self.current += 1
        return value

# Цикл for под капотом сам вызывает __iter__(), а затем __next__() пока не поймает StopIteration
for num in MyRange(1, 4):
    print(num)

1
2
3


#### Задание

Напишите класс-итератор `EvenNumbers`, который при инициализации принимает список чисел, а при итерации по нему возвращает **только четные** числа из этого списка.

Обязательно используйте методы `__iter__` и `__next__`.

In [27]:
class EvenNumbers:
    def __init__(self, numbers):
        self.numbers = numbers
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        while self.index < len(self.numbers):
            value = self.numbers[self.index]
            self.index += 1
            if value % 2 == 0:
                return value
        raise StopIteration

numbers = [1, 2, 3, 4, 5, 6]
even_iter = EvenNumbers(numbers)


assert list(even_iter) == [2, 4, 6]

### Генераторы `yield`

Самый простой способ создать свой итератор — написать генератор. Это обычная функция, но вместо `return` в ней используется слово `yield`. 

В момент `yield` функция «ставится на паузу», возвращает значение и ждет. При следующем обращении (через цикл `for` или `next()`) она продолжает работу с того же места.

**Как генераторы оптимизируют память?**
Представьте, что вам нужно обработать лог-файл размером 5 Гигабайт. Если вы попытаетесь прочитать его целиком в список строк (`lines = file.readlines()`), ваша программа заберет 5 Гб оперативной памяти и может упасть.
Генератор же позволяет читать и отдавать по одной строке за раз. В памяти одновременно находится только одна текущая строка, а потребление памяти остается на уровне пары мегабайт, независимо от размера файла!

In [28]:
def count_up_to(max_val):
    count = 1
    while count <= max_val:
        yield count
        count += 1

counter = count_up_to(3)
for num in counter:
    print(num)

1
2
3


#### Задание

Напишите генератор `fibonacci(n)`, который будет выдавать первые `n` чисел Фибоначчи. (Они начинаются с 0 и 1, а каждое следующее равно сумме двух предыдущих: 0, 1, 1, 2, 3, 5, 8...)

In [29]:
def fibonacci(n):
    a = 0
    b = 1
    for _ in range(n):
        yield a
        a, b = b, a + b

assert list(fibonacci(7)) == [0, 1, 1, 2, 3, 5, 8]

### Генераторные выражения (Generator Comprehensions)

Вы уже знаете про List Comprehensions (генераторы списков), которые пишутся в квадратных скобках `[x for x in ...]`. Они создают список целиком.

Если заменить квадратные скобки на круглые `(x for x in ...)`, получится **генераторное выражение**. Оно делает то же самое, но возвращает объект-генератор, который вычисляет элементы «лениво» (по одному), экономя память.

In [8]:
import sys # нужно для sizeof (проверка памяти)

# Список: вычисляет квадраты 100 000 чисел и сохраняет их в памяти
list_comp = [x**2 for x in range(100000)]
print(f"Память под список: {sys.getsizeof(list_comp)} байт")

# Генератор: не вычисляет ничего заранее, просто готов отдавать квадраты по одному
gen_comp = (x**2 for x in range(100000))
print(f"Память под генератор: {sys.getsizeof(gen_comp)} байт")

# Мы можем пройтись по нему, как обычно
print("Первый элемент:", next(gen_comp))

Память под список: 800984 байт
Память под генератор: 200 байт
Первый элемент: 0


#### Задание

У вас есть огромный список чисел от 1 до миллиона (он уже создан). Вам нужно найти сумму квадратов всех **нечетных** чисел из этого списка.

Напишите решение в одну строку, используя функцию `sum()` и **генераторное выражение** (в круглых скобках), чтобы не создавать в памяти промежуточный список с квадратами нечетных чисел.

In [30]:
huge_numbers = range(1, 1000001)  # Исходные данные (range тоже ленивый)

# TODO ваш код здесь (используйте sum и генераторное выражение)
total_sum = sum((i for i in huge_numbers if i % 2 == 0))

print(total_sum) # Должно получиться очень большое число :)

250000500000


### `if __name__ == '__main__'`

В реальных проектах вы часто будете видеть эту конструкцию внизу файлов. Пугаться не надо.
В Python каждый файл (модуль) имеет специальную встроенную переменную `__name__`:
- Если вы запускаете этот файл **напрямую** (например, нажав Run в IDE или написав `python script.py`), то переменная `__name__` равна строке `"__main__"`.
- Если же этот файл кто-то **импортирует** (`import script`), то `__name__` будет равна имени файла (`"script"`).

Эта конструкция нужна, чтобы какой-то код выполнялся только при прямом запуске, и не выполнялся случайно при импорте функций из этого файла.

Я этим не пользуюсь потому что у меня строго разграничены файлы откуда что-то импортируется, и файлы которые запускаются.

In [ ]:
# Пример того, как это выглядит в обычных .py файлах:

def some_useful_function():
    return 42

if __name__ == '__main__':
    print('Скрипт запущен напрямую!')
    print('Результат:', some_useful_function())